# 01 — Drug Repurposing Hub (DRH)

Map JUMP-CP compounds into the Broad Institute Drug Repurposing Hub.

- **Source**: Corsello et al., Nature Medicine 2017 
- **URL**: https://repo-hub.broadinstitute.org/repurposing 
- **Version**: March 24, 2020 (`repurposing_drugs_20200324.txt`) 
- **Identifier**: InChIKey (direct match) 
- **Curation**: Expert-curated compound identities + literature-reported MOA/target 
- **Confidence metric**: None (all entries are curated; implicit confidence = 1.0)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path("../..")  # repo root
CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)

# Standardized schema columns (same for ALL databases)
STANDARD_COLS = [
    "Metadata_InChIKey",   # Chemical identifier
    "Metadata_JCP2022",    # JUMP compound ID
    "source",              # Database name
    "label_target",        # Gene symbol (e.g., EGFR)
    "label_moa",           # MOA (e.g., EGFR inhibitor)
    "confidence",          # Normalized 0-1
    "evidence_count",      # N independent sources
    "is_primary",          # Primary curated annotation?
]

## 1. Load Data

In [ ]:
# Load JUMP compound registry (from notebook 00)
compounds = pd.read_parquet(CACHE / "jump_compounds.parquet")
print(f"JUMP compounds: {len(compounds):,}")
print(compounds["plate_type"].value_counts().to_string())

In [ ]:
# Load DRH MOA annotations (exploded: one row per compound x MOA)
drh_moa = pd.read_csv(ROOT / "inputs/metadata/repurposinghub_inchi_to_moa_moaexploded.csv")
print(f"DRH MOA rows: {len(drh_moa):,}")
print(f"Unique InChIKeys: {drh_moa['Metadata_InChIKey'].nunique():,}")
print(f"Unique MOAs: {drh_moa['Metadata_DRH_MOA'].nunique():,}")
print()

# Load DRH Target annotations (exploded: one row per compound x target gene)
drh_tgt = pd.read_csv(ROOT / "inputs/metadata/repurposinghub_inchi_to_moa_targetexploded.csv")
print(f"DRH Target rows: {len(drh_tgt):,}")
print(f"Unique InChIKeys: {drh_tgt['Metadata_InChIKey'].nunique():,}")
print(f"Unique targets: {drh_tgt['Metadata_DRH_TARGET'].nunique():,}")

## 2. Map JUMP Compounds to DRH

Join on InChIKey (direct match, no bridging needed).

In [ ]:
# Map: JUMP compounds -> DRH MOA
mapped_moa = compounds.merge(
    drh_moa[["Metadata_InChIKey", "Metadata_DRH_MOA", "Metadata_target",
             "Metadata_pert_iname", "Metadata_clinical_phase"]],
    on="Metadata_InChIKey",
    how="inner",
)
print(f"Compounds with DRH MOA: {mapped_moa['Metadata_InChIKey'].nunique():,} / {len(compounds):,}")
print(f"Coverage: {mapped_moa['Metadata_InChIKey'].nunique() / len(compounds) * 100:.1f}%")

In [ ]:
# Map: JUMP compounds -> DRH Target
mapped_tgt = compounds.merge(
    drh_tgt[["Metadata_InChIKey", "Metadata_DRH_TARGET", "Metadata_moa",
             "Metadata_pert_iname", "Metadata_clinical_phase"]],
    on="Metadata_InChIKey",
    how="inner",
)
print(f"Compounds with DRH Target: {mapped_tgt['Metadata_InChIKey'].nunique():,} / {len(compounds):,}")
print(f"Coverage: {mapped_tgt['Metadata_InChIKey'].nunique() / len(compounds) * 100:.1f}%")

## 3. Coverage Metrics

### 3.1 Raw Coverage

In [ ]:
# Coverage by plate type
def coverage_report(mapped_df, compounds_df, label_col, id_col="Metadata_InChIKey"):
    """Compute coverage metrics for a mapped annotation DataFrame."""
    results = []
    for plate_type in ["TARGET2", "COMPOUND", "ALL"]:
        if plate_type == "ALL":
            cpd_sub = compounds_df
            map_sub = mapped_df
        else:
            cpd_sub = compounds_df[compounds_df["plate_type"] == plate_type]
            map_sub = mapped_df[mapped_df["plate_type"] == plate_type]

        n_total = cpd_sub[id_col].nunique()
        n_mapped = map_sub[id_col].nunique()
        n_labels = map_sub[label_col].nunique() if len(map_sub) > 0 else 0

        results.append({
            "plate_type": plate_type,
            "n_total": n_total,
            "n_mapped": n_mapped,
            "coverage_pct": n_mapped / n_total * 100 if n_total > 0 else 0,
            "n_unique_labels": n_labels,
        })
    return pd.DataFrame(results)

print("=== MOA Coverage ===")
display(coverage_report(mapped_moa, compounds, "Metadata_DRH_MOA"))
print()
print("=== Target Coverage ===")
display(coverage_report(mapped_tgt, compounds, "Metadata_DRH_TARGET"))

### 3.2 Evaluable Coverage (groups with >= N compounds)

In [ ]:
def evaluable_coverage(mapped_df, compounds_df, label_col, id_col="Metadata_InChIKey",
                       thresholds=(2, 3, 5, 10)):
    """Coverage after filtering to groups with >= N compounds."""
    results = []
    for plate_type in ["TARGET2", "COMPOUND", "ALL"]:
        if plate_type == "ALL":
            map_sub = mapped_df
        else:
            map_sub = mapped_df[mapped_df["plate_type"] == plate_type]

        # Deduplicate: one row per compound x label
        dedup = map_sub.drop_duplicates(subset=[id_col, label_col])

        for min_n in thresholds:
            group_sizes = dedup.groupby(label_col)[id_col].nunique()
            valid_groups = group_sizes[group_sizes >= min_n]
            valid_labels = valid_groups.index
            valid_compounds = dedup[dedup[label_col].isin(valid_labels)][id_col].nunique()

            results.append({
                "plate_type": plate_type,
                "min_compounds": min_n,
                "n_groups": len(valid_groups),
                "n_evaluable_compounds": valid_compounds,
                "median_group_size": valid_groups.median() if len(valid_groups) > 0 else 0,
                "max_group_size": valid_groups.max() if len(valid_groups) > 0 else 0,
            })
    return pd.DataFrame(results)

print("=== MOA Evaluable Coverage ===")
display(evaluable_coverage(mapped_moa, compounds, "Metadata_DRH_MOA"))
print()
print("=== Target Evaluable Coverage ===")
display(evaluable_coverage(mapped_tgt, compounds, "Metadata_DRH_TARGET"))

## 4. Annotation Quality Metrics

### 4.1 Confidence Distribution

DRH has no explicit confidence tiers (all entries are expert-curated),
but we can use `clinical_phase` as a proxy for compound characterization depth.

In [ ]:
# Clinical phase distribution as confidence proxy
phase_map = {
    "Launched": 1.0,
    "Phase 3": 0.75,
    "Phase 2": 0.60,
    "Phase 1": 0.50,
    "Preclinical": 0.25,
}

# Per-compound (deduplicated)
per_cpd = mapped_moa.drop_duplicates(subset="Metadata_InChIKey")
per_cpd["confidence"] = per_cpd["Metadata_clinical_phase"].map(phase_map).fillna(0.25)

print("Clinical phase distribution:")
print(per_cpd["Metadata_clinical_phase"].value_counts().to_string())
print()
print("Confidence distribution:")
print(per_cpd["confidence"].describe().to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
per_cpd["Metadata_clinical_phase"].value_counts().plot.bar(ax=axes[0], title="Clinical Phase")
axes[0].set_ylabel("Compounds")
per_cpd["confidence"].hist(bins=20, ax=axes[1])
axes[1].set_title("Confidence (from clinical phase)")
axes[1].set_xlabel("Confidence")
plt.tight_layout()
plt.savefig(CACHE / "drh_confidence_distribution.pdf", bbox_inches="tight")
plt.show()

### 4.2 Polypharmacology (Multi-target / Multi-MOA)

In [ ]:
# How many targets per compound?
targets_per_cpd = (
    mapped_tgt.groupby("Metadata_InChIKey")["Metadata_DRH_TARGET"]
    .nunique()
    .rename("n_targets")
)
print("Targets per compound:")
print(targets_per_cpd.describe().to_string())
print(f"\nSingle-target: {(targets_per_cpd == 1).sum()} ({(targets_per_cpd == 1).mean()*100:.1f}%)")
print(f"Multi-target (2+): {(targets_per_cpd >= 2).sum()} ({(targets_per_cpd >= 2).mean()*100:.1f}%)")
print(f"Highly poly (5+): {(targets_per_cpd >= 5).sum()} ({(targets_per_cpd >= 5).mean()*100:.1f}%)")

# How many MOAs per compound?
moas_per_cpd = (
    mapped_moa.groupby("Metadata_InChIKey")["Metadata_DRH_MOA"]
    .nunique()
    .rename("n_moas")
)
print(f"\nMOAs per compound:")
print(moas_per_cpd.describe().to_string())
print(f"\nSingle-MOA: {(moas_per_cpd == 1).sum()} ({(moas_per_cpd == 1).mean()*100:.1f}%)")
print(f"Multi-MOA (2+): {(moas_per_cpd >= 2).sum()} ({(moas_per_cpd >= 2).mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
targets_per_cpd.clip(upper=15).hist(bins=15, ax=axes[0])
axes[0].set_title("Targets per compound (DRH)")
axes[0].set_xlabel("N targets")
moas_per_cpd.clip(upper=5).hist(bins=5, ax=axes[1])
axes[1].set_title("MOAs per compound (DRH)")
axes[1].set_xlabel("N MOAs")
plt.tight_layout()
plt.savefig(CACHE / "drh_polypharmacology.pdf", bbox_inches="tight")
plt.show()

### 4.3 Label Group Size Distribution

In [ ]:
def plot_group_sizes(mapped_df, label_col, title, ax):
    """Plot distribution of compounds per label group."""
    dedup = mapped_df.drop_duplicates(subset=["Metadata_InChIKey", label_col])
    sizes = dedup.groupby(label_col)["Metadata_InChIKey"].nunique().sort_values(ascending=False)
    sizes.clip(upper=50).hist(bins=25, ax=ax)
    ax.axvline(x=3, color="red", linestyle="--", label="min=3 threshold")
    ax.set_title(f"{title} (n={len(sizes)} groups)")
    ax.set_xlabel("Compounds per group")
    ax.set_ylabel("N groups")
    ax.legend()
    return sizes

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
moa_sizes = plot_group_sizes(mapped_moa, "Metadata_DRH_MOA", "MOA group sizes", axes[0])
tgt_sizes = plot_group_sizes(mapped_tgt, "Metadata_DRH_TARGET", "Target group sizes", axes[1])
plt.tight_layout()
plt.savefig(CACHE / "drh_group_sizes.pdf", bbox_inches="tight")
plt.show()

print(f"\nMOA: {len(moa_sizes)} total, {(moa_sizes >= 3).sum()} with >= 3 cpds, {(moa_sizes >= 5).sum()} with >= 5 cpds")
print(f"Target: {len(tgt_sizes)} total, {(tgt_sizes >= 3).sum()} with >= 3 cpds, {(tgt_sizes >= 5).sum()} with >= 5 cpds")

## 5. Morphological Predictivity

The key test: do DRH labels predict morphological similarity in Cell Painting data?

We use a well-corrected scenario (S1 or S2) and compute mAP using DRH labels
as the grouping variable. Higher mAP = labels better predict morphology.

In [ ]:
import anndata as ad
import sys
sys.path.insert(0, str(ROOT))

# Load S2 (3 sources, TARGET2 only — well-understood scenario)
adata = ad.read_h5ad(ROOT / "outputs/scenario_2/mad_int_featselect_all_methods.h5ad")
print(f"S2: {adata.n_obs} cells, {adata.n_vars} features")
print(f"Methods: {list(adata.obsm.keys())}")
print(f"Compounds: {adata.obs['Metadata_JCP2022'].nunique()}")

In [ ]:
# Merge DRH MOA labels into adata
moa_lookup = (
    mapped_moa[["Metadata_InChIKey", "Metadata_DRH_MOA"]]
    .drop_duplicates()
    .groupby("Metadata_InChIKey")["Metadata_DRH_MOA"]
    .first()  # Take first MOA for multi-MOA compounds (simplification)
    .reset_index()
)

# Merge via InChIKey
adata.obs = adata.obs.reset_index(drop=True)
adata.obs = adata.obs.merge(moa_lookup, on="Metadata_InChIKey", how="left")

# Also merge DRH Target
tgt_lookup = (
    mapped_tgt[["Metadata_InChIKey", "Metadata_DRH_TARGET"]]
    .drop_duplicates()
    .groupby("Metadata_InChIKey")["Metadata_DRH_TARGET"]
    .first()  # Take first/primary target
    .reset_index()
)
adata.obs = adata.obs.merge(tgt_lookup, on="Metadata_InChIKey", how="left")

print(f"Compounds with DRH MOA: {adata.obs['Metadata_DRH_MOA'].notna().sum()} / {len(adata.obs)}")
print(f"Compounds with DRH Target: {adata.obs['Metadata_DRH_TARGET'].notna().sum()} / {len(adata.obs)}")
print(f"Unique MOAs in S2: {adata.obs['Metadata_DRH_MOA'].nunique()}")
print(f"Unique Targets in S2: {adata.obs['Metadata_DRH_TARGET'].nunique()}")

In [ ]:
from copairs.map import average_precision, mean_average_precision

def compute_map_for_label(adata, embedding_key, label_col, min_group_size=3):
    """Compute mAP using a given label column as the grouping variable."""
    # Filter to cells with labels
    mask = adata.obs[label_col].notna()
    obs = adata.obs[mask].copy()
    emb = adata.obsm[embedding_key][mask]

    # Filter to groups with >= min_group_size compounds
    group_sizes = obs.groupby(label_col)["Metadata_JCP2022"].nunique()
    valid_groups = group_sizes[group_sizes >= min_group_size].index
    mask2 = obs[label_col].isin(valid_groups)
    obs = obs[mask2].reset_index(drop=True)
    emb = emb[mask2]

    if len(obs) == 0 or obs[label_col].nunique() < 2:
        return {"mean_map": np.nan, "n_compounds": 0, "n_groups": 0}

    # Compute per-compound average precision
    result = average_precision(
        obs,
        emb,
        pos_sameby=[label_col],
        neg_diffby=[label_col],
        batch_size=None,
    )

    map_result = mean_average_precision(result, threshold=0.05)

    return {
        "mean_map": result["average_precision"].mean(),
        "median_map": result["average_precision"].median(),
        "frac_retrievable": (result["average_precision"] > 0).mean(),
        "n_compounds": obs["Metadata_JCP2022"].nunique(),
        "n_groups": obs[label_col].nunique(),
        "n_cells": len(obs),
    }

print("Function defined. Computing mAP for each method x label level...")

In [ ]:
# Compute mAP for each embedding method x label level
results = []
methods = [k for k in adata.obsm.keys() if not k.startswith("X_")]

for method in methods:
    for label_col, label_level in [
        ("Metadata_DRH_MOA", "moa"),
        ("Metadata_DRH_TARGET", "target"),
    ]:
        try:
            r = compute_map_for_label(adata, method, label_col, min_group_size=3)
            r["method"] = method
            r["label_level"] = label_level
            r["database"] = "drh"
            results.append(r)
            print(f"{method:20s} | {label_level:6s} | mAP={r['mean_map']:.3f} | {r['n_groups']} groups, {r['n_compounds']} cpds")
        except Exception as e:
            print(f"{method:20s} | {label_level:6s} | ERROR: {e}")

results_df = pd.DataFrame(results)
results_df.to_parquet(CACHE / "drh_morphological_predictivity.parquet", index=False)

In [ ]:
# Plot: mAP by method and label level
fig, ax = plt.subplots(figsize=(14, 5))
pivot = results_df.pivot(index="method", columns="label_level", values="mean_map").sort_values("moa", ascending=True)
pivot.plot.barh(ax=ax)
ax.set_xlabel("Mean Average Precision (mAP)")
ax.set_title("DRH: Morphological Predictivity by Label Level (S2)")
ax.legend(title="Label level")
plt.tight_layout()
plt.savefig(CACHE / "drh_morphological_predictivity.pdf", bbox_inches="tight")
plt.show()

## 6. Export Standardized Annotations

Save in the common schema used across all database notebooks.

In [ ]:
# Build standardized annotation table
# One row per compound x target (most granular level)
std = mapped_tgt[["Metadata_InChIKey", "Metadata_JCP2022", "Metadata_DRH_TARGET",
                   "Metadata_clinical_phase", "plate_type"]].copy()
std = std.rename(columns={"Metadata_DRH_TARGET": "label_target"})

# Add MOA by merging back
moa_per_cpd = (
    mapped_moa[["Metadata_InChIKey", "Metadata_DRH_MOA"]]
    .drop_duplicates()
    .groupby("Metadata_InChIKey")["Metadata_DRH_MOA"]
    .first()
    .reset_index()
    .rename(columns={"Metadata_DRH_MOA": "label_moa"})
)
std = std.merge(moa_per_cpd, on="Metadata_InChIKey", how="left")

# Add standardized fields
std["source"] = "drh"
std["confidence"] = std["Metadata_clinical_phase"].map(phase_map).fillna(0.25)
std["evidence_count"] = 1  # DRH is binary: curated or not
std["is_primary"] = True   # All DRH annotations are curated/primary

# Select standard columns
std = std[[c for c in STANDARD_COLS if c in std.columns] + ["plate_type"]]

print(f"Standardized DRH annotations: {len(std):,} rows")
print(f"Unique compounds: {std['Metadata_InChIKey'].nunique():,}")
print(f"Unique targets: {std['label_target'].nunique():,}")
print(f"Unique MOAs: {std['label_moa'].nunique():,}")

std.to_parquet(CACHE / "drh_annotations.parquet", index=False)
print("\nSaved to cache/drh_annotations.parquet")
std.head(10)

## 7. Summary Statistics (for cross-database comparison)

These metrics will be collected from all database notebooks into a single comparison table.

In [ ]:
summary = {
    "database": "drh",
    "version": "2020-03-24",
    "identifier_bridge": "direct (InChIKey)",
    "bridging_loss_pct": 0.0,
    "license": "open",
    "last_updated": "2020-03-24",
    # Raw coverage
    "n_compounds_mapped_target2": mapped_moa[mapped_moa["plate_type"] == "TARGET2"]["Metadata_InChIKey"].nunique(),
    "n_compounds_mapped_compound": mapped_moa[mapped_moa["plate_type"] == "COMPOUND"]["Metadata_InChIKey"].nunique(),
    "coverage_pct_target2": mapped_moa[mapped_moa["plate_type"] == "TARGET2"]["Metadata_InChIKey"].nunique() / (compounds["plate_type"] == "TARGET2").sum() * 100,
    "coverage_pct_compound": mapped_moa[mapped_moa["plate_type"] == "COMPOUND"]["Metadata_InChIKey"].nunique() / (compounds["plate_type"] == "COMPOUND").sum() * 100,
    # Evaluable (>= 3 cpds/group)
    "n_moa_groups_ge3": (moa_sizes >= 3).sum(),
    "n_target_groups_ge3": (tgt_sizes >= 3).sum(),
    # Polypharmacology
    "pct_single_target": (targets_per_cpd == 1).mean() * 100,
    "pct_multi_moa": (moas_per_cpd >= 2).mean() * 100,
    "median_targets_per_cpd": targets_per_cpd.median(),
    # Confidence
    "has_confidence_metric": False,
    "confidence_metric_name": "clinical_phase (proxy)",
    # Label levels available
    "has_target_labels": True,
    "has_moa_labels": True,
    "has_pathway_labels": False,
}

summary_df = pd.DataFrame([summary])
summary_df.to_parquet(CACHE / "drh_summary.parquet", index=False)
print("Summary saved. Key metrics:")
for k, v in summary.items():
    print(f"  {k}: {v}")